## Exercise 1: Set Up the AutoSphere Data Platform

You are a data engineer at **AutoSphere AG**, a global automotive manufacturer. Your connected vehicle platform ingests telemetry from millions of vehicles, manages customer registration data, and tracks service history.

Before you can apply governance controls, you need to create the catalog, schema, and tables that form the foundation of the platform. You'll also add **comments** to document the purpose of each table and column — making your data assets discoverable by other teams.

### Setup — Create the catalog, schema, and tables

To keep the focus on governance controls, the AutoSphere data platform is set up for you in the cell below. It creates:

- **`automotive_catalog`** — the Unity Catalog catalog for the AutoSphere platform, with a descriptive comment
- **`automotive_catalog.governance_lab`** — the schema used throughout this lab
- **`customer_registrations`** — one row per registered customer and their associated vehicle; includes a table-level comment
- **`vehicle_telemetry`** — continuous telemetry events from connected vehicles; includes column-level comments on `speed_kmh` and `battery_level_pct`

Run the cell to prepare your environment, then continue to Exercise 2.

In [ ]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
# Use the current catalog to reliably find the workspace default catalog,
# regardless of its naming convention.
default_catalog = spark.catalog.currentCatalog()

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS automotive_catalog
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Connected vehicle platform catalog for AutoSphere AG — stores registration, telemetry, and service data.'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS automotive_catalog.governance_lab
  COMMENT 'Governance lab schema for hands-on Unity Catalog exercises.';

-- Create the customer_registrations table
CREATE TABLE IF NOT EXISTS automotive_catalog.governance_lab.customer_registrations
  COMMENT 'One row per registered AutoSphere customer and their associated vehicle.'
AS SELECT * FROM VALUES
  (1, 'Lukas Bauer',    'lukas.bauer@autosphere.de',   'DE-LB-4821', 'DE', 'VH-001'),
  (2, 'Sophie Martin',  'sophie.martin@autosphere.fr',  'FR-SM-9034', 'FR', 'VH-002'),
  (3, 'James Clarke',   'james.clarke@autosphere.uk',   'UK-JC-1172', 'UK', 'VH-003'),
  (4, 'Yuki Tanaka',    'yuki.tanaka@autosphere.jp',    'JP-YT-5561', 'JP', 'VH-004'),
  (5, 'Maria Santos',   'maria.santos@autosphere.pt',   'PT-MS-8843', 'PT', 'VH-005'),
  (6, 'Carlos Rivera',  'carlos.rivera@autosphere.es',  'ES-CR-3309', 'ES', 'VH-006')
AS t(customer_id, full_name, email, driver_license_no, country, vehicle_id);

-- Create the vehicle_telemetry table
CREATE TABLE IF NOT EXISTS automotive_catalog.governance_lab.vehicle_telemetry (
  event_id          BIGINT    COMMENT 'Unique telemetry event identifier',
  vehicle_id        STRING,
  event_time        TIMESTAMP,
  speed_kmh         INT       COMMENT 'Instantaneous vehicle speed in kilometres per hour',
  battery_level_pct INT       COMMENT 'State of charge as a percentage (0–100)',
  latitude          DOUBLE,
  longitude         DOUBLE,
  country           STRING
);

INSERT INTO automotive_catalog.governance_lab.vehicle_telemetry VALUES
  (1, 'VH-001', TIMESTAMP '2026-03-01 08:15:00', 112, 78,  48.8566,   2.3522, 'DE'),
  (2, 'VH-002', TIMESTAMP '2026-03-01 08:16:00', 95,  55,  51.5074,  -0.1278, 'FR'),
  (3, 'VH-003', TIMESTAMP '2026-03-01 08:17:00', 130, 91,  40.4168,  -3.7038, 'UK'),
  (4, 'VH-004', TIMESTAMP '2026-03-01 08:18:00', 88,  34,  35.6762, 139.6503, 'JP'),
  (5, 'VH-005', TIMESTAMP '2026-03-01 08:19:00', 105, 62,  38.7223,  -9.1393, 'PT'),
  (6, 'VH-006', TIMESTAMP '2026-03-01 08:20:00', 77,  47,  41.3851,   2.1734, 'ES');

## Exercise 2: Tag Data Assets for Governance

AutoSphere's compliance team requires that all tables and columns containing **personally identifiable information (PII)** are consistently labelled. Tags enable automated discovery and will later drive access policies.

In this exercise, you apply table-level and column-level tags to classify the `customer_registrations` table.

### Task 2.1 — Add table-level tags

Tag the `customer_registrations` table to classify it by domain and sensitivity:

- `domain` = `customer`
- `data_classification` = `confidential`

> 🤖 **Genie Code tip:** Ask:
> *"How do I add key-value tags to a Unity Catalog table in Databricks SQL?"*

**Hint:** Use `ALTER TABLE ... SET TAGS ('key' = 'value', 'key2' = 'value2')`.

In [ ]:
%sql
-- TODO: Add table-level tags to automotive_catalog.governance_lab.customer_registrations
-- Tags: domain = customer, data_classification = confidential

### Task 2.2 — Add column-level PII tags

Tag the PII columns in `customer_registrations` to identify them as sensitive:

- Column `email` → `pii` = `email`
- Column `driver_license_no` → `pii` = `driver_license`

> 🤖 **Genie Code tip:** Ask:
> *"How do I add a tag to a specific column in a Unity Catalog table using SQL in Databricks?"*

**Hint:** Use `ALTER TABLE ... ALTER COLUMN <col> SET TAGS (...)`.

In [ ]:
%sql
-- TODO: Tag the email column with pii = email

-- TODO: Tag the driver_license_no column with pii = driver_license

### Task 2.3 — Verify the tags

Use `system.information_schema.table_tags` to confirm the table-level tags have been applied. Then describe the columns to confirm the column tags are present. Use `system.information_schema.column_tags` for this.

> 🤖 **Genie Code tip:** Ask:
> *"How can I verify tags on a Unity Catalog table and its columns in Databricks SQL?"*

In [ ]:
%sql
-- Show table-level tags for customer_registrations

In [ ]:
%sql
-- Show column-level tags for customer_registrations

## Exercise 3: Configure Data Retention

AutoSphere's vehicle telemetry table is written to continuously. To comply with GDPR data minimisation principles and control storage costs, the platform must:

- Retain only **14 days** of table history.
- Permanently remove old data files using `VACUUM`.
- Enable **predictive optimization** so that future maintenance runs automatically.

In this exercise, you configure retention settings, simulate record deletion, and run a `VACUUM` operation.

### Task 3.1 — Configure retention properties

Set the following Delta Lake table properties on `vehicle_telemetry`:

- `delta.logRetentionDuration` = `interval 14 days`
- `delta.deletedFileRetentionDuration` = `interval 14 days`

> 🤖 **Genie Code tip:** Ask:
> *"How do I set Delta Lake retention duration properties on a Unity Catalog table in Databricks?"*

**Hint:** Use `ALTER TABLE ... SET TBLPROPERTIES (...)`.

In [ ]:
%sql
-- TODO: Set logRetentionDuration and deletedFileRetentionDuration to 14 days
-- on automotive_catalog.governance_lab.vehicle_telemetry

### Task 3.2 — Simulate a GDPR deletion request

A customer has submitted a right-to-erasure request. Delete all telemetry records for vehicle `VH-003` from the `vehicle_telemetry` table.

After deletion, verify the records are gone by querying the table.

> 🤖 **Genie Code tip:** Ask:
> *"How do I delete specific rows from a Delta Lake table in Databricks SQL?"*

In [ ]:
%sql
-- TODO: Delete all rows for vehicle_id = 'VH-003' from vehicle_telemetry

-- TODO: Verify the deletion by querying the table

### Task 3.3 — Run VACUUM to purge deleted data files

The deleted rows still exist in underlying Parquet files until `VACUUM` is run. Execute `VACUUM` on `vehicle_telemetry` to permanently remove files that are no longer referenced by any current or recent table version.

Use the default retention of **168 hours (7 days)**. This is safe, works on Serverless compute, and is appropriate for a lab environment.

> 🤖 **Genie Code tip:** Ask:
> *"How do I run VACUUM on a Delta Lake table in Databricks, and what does the RETAIN option do?"*

**Note:** On Serverless compute, overriding the minimum retention duration is not supported. Always specify a `RETAIN` value of at least 168 hours, or omit `RETAIN` entirely to use the default.

In [ ]:
%sql
-- TODO: Run VACUUM on automotive_catalog.governance_lab.vehicle_telemetry
-- Use RETAIN 168 HOURS (7 days) or omit RETAIN to use the default

### Task 3.4 — Enable predictive optimization

Enable **predictive optimization** on the `governance_lab` schema so that future `VACUUM` and `OPTIMIZE` maintenance runs automatically without manual scheduling.

> 🤖 **Genie Code tip:** Ask:
> *"How do I enable predictive optimization on a Unity Catalog schema in Databricks?"*

**Hint:** Use `ALTER SCHEMA ... ENABLE PREDICTIVE OPTIMIZATION`.

In [ ]:
%sql
-- TODO: Enable predictive optimization on automotive_catalog.governance_lab

## Exercise 4: Query Data Lineage Programmatically

> 📋 **Before running this exercise**: If you haven't already done so, pause here and follow the **Catalog Explorer lineage steps** in the lab setup instructions to explore lineage visually in the UI. Then return here to query lineage data using SQL.

Unity Catalog captures all read and write events and stores them in the `system.access.table_lineage` system table. This lets you programmatically answer questions like: *"Which tables in our automotive catalog were most queried this week?"*

### Task 4.1 — Find recently accessed tables in your catalog

Query `system.access.table_lineage` to find all **write events** (source tables) within the past 7 days that relate to `automotive_catalog`. Return the table name and the count of distinct events, ordered by the most active tables first.

> 🤖 **Genie Code tip:** Ask:
> *"How do I query system.access.table_lineage to find recently accessed Unity Catalog tables in Databricks?"*

**Hint:** Filter on `source_table_full_name LIKE 'automotive_catalog%'` and group by table name.

In [ ]:
%sql
-- TODO: Query system.access.table_lineage for tables in automotive_catalog accessed in the last 7 days
-- Return: source_table_full_name, count of events
-- Order by most active first

### Task 4.2 — View the history of vehicle_telemetry

Use `DESCRIBE HISTORY` to see the full Delta Lake operation history for `vehicle_telemetry`. Identify the version where the deletion occurred (`DELETE` operation).

> 🤖 **Genie Code tip:** Ask:
> *"How do I view the full version history of a Delta Lake table in Databricks SQL?"*

In [ ]:
%sql
-- TODO: Describe the history of automotive_catalog.governance_lab.vehicle_telemetry

## Exercise 5: Audit Logging

AutoSphere's compliance team needs to prove to regulators that only authorised users accessed customer data during the past 30 days. The team also wants to be alerted whenever permissions are modified on the Unity Catalog metastore.

You will use the `system.access.audit` system table to answer both of these requirements.

### Task 5.1 — Find who accessed the customer_registrations table

Query `system.access.audit` to list all users who accessed `automotive_catalog.governance_lab.customer_registrations` in the past 30 days. Include the action name and the time of each access, ordered by most recent first.

> 🤖 **Genie Code tip:** Ask:
> *"How do I query the Databricks audit log system table to find who accessed a specific Unity Catalog table?"*

**Hint:** Filter on `request_params.full_name_arg` and `action_name IN ('getTable', 'createTable')`. Use `user_identity.email` for the user column.

In [ ]:
%sql
-- TODO: Query system.access.audit for access events on automotive_catalog.governance_lab.customer_registrations
-- Return: user email, action_name, event_time
-- Filter to the past 30 days, order by event_time DESC

### Task 5.2 — Detect permission changes across your catalog

Query `system.access.audit` to list all **permission change events** (`updatePermissions` action) on securable objects in `automotive_catalog` during the past 30 days. Include who made the change and what object was affected.

> 🤖 **Genie Code tip:** Ask:
> *"How do I find all permission changes in the Databricks audit log for a specific Unity Catalog catalog?"*

**Hint:** Filter on `service_name = 'unityCatalog'`, `action_name = 'updatePermissions'`, and `request_params.securable_full_name LIKE 'automotive_catalog%'`.

In [ ]:
%sql
-- TODO: Query system.access.audit for permission change events on automotive_catalog objects
-- Return: event_time, user email, securable_type, securable_full_name, changes
-- Filter to the past 30 days, order by event_time DESC

### Task 5.3 — Summarise audit activity by user

Create a summary query that counts the total number of audit events per user for Unity Catalog actions in the past 7 days. This gives the compliance team a quick overview of who was most active on the platform.

> 🤖 **Genie Code tip:** Ask:
> *"How do I aggregate the Databricks audit log to count events per user for Unity Catalog activity?"*

In [ ]:
%sql
-- TODO: Count total audit events per user from system.access.audit
-- Filter to service_name = 'unityCatalog' and the last 7 days
-- Return: user email, event count
-- Order by most active user first